# 01 — Exploratory Data Analysis (EDA) — UNSW-NB15 (Mode 1)

This notebook provides the exploratory analysis of the **cleaned UNSW-NB15 dataset** for **Mode 1** of the Network Intrusion Detection System (offline held-out evaluation).

### Scope & Constraints
- Uses cleaned interim Parquet files (`data/interim/UNSW_NB15_training_clean.parquet` and `data/interim/UNSW_NB15_testing_clean.parquet`).
- Strictly preserves the official train/test split without data leakage.
- Excludes non-feature columns (`id`, `label`, `attack_cat`) from the model input space.
- Investigates target distributions, multicollinearity, categorical domain shift, and rare attack classes.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure src is on Python path
ROOT_DIR = Path("..").resolve()
sys.path.append(str(ROOT_DIR / "src"))

from nids.config import (
    CLEAN_TRAIN_PARQUET, CLEAN_TEST_PARQUET,
    TARGET_BINARY, TARGET_MULTI, NON_FEATURE_COLUMNS
)
from nids.eda import (
    load_clean_data, analyze_target_distributions,
    analyze_categorical_features, analyze_numeric_features,
    analyze_correlations, analyze_rare_categories
)

sns.set_theme(style="whitegrid")
print("Imports successful. Environment configured.")

## 1. Load Cleaned Datasets
We load the verified and cleaned Parquet files from `data/interim/`.

In [ ]:
df_train, df_test = load_clean_data(ROOT_DIR / "data/interim/UNSW_NB15_training_clean.parquet", ROOT_DIR / "data/interim/UNSW_NB15_testing_clean.parquet")
print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape:  {df_test.shape}")
df_train.head(3)

## 2. Target Distributions (Binary & Multiclass)
Examining the class balance for binary classification (`label`) and multiclass classification (`attack_cat`).

In [ ]:
target_stats = analyze_target_distributions(df_train, df_test)
print("Binary targets (Train):")
print(target_stats["binary"]["train"])
print("\nBinary targets (Test):")
print(target_stats["binary"]["test"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tr_bin = df_train[TARGET_BINARY].value_counts().rename(index={0: "Normal (0)", 1: "Attack (1)"})
te_bin = df_test[TARGET_BINARY].value_counts().rename(index={0: "Normal (0)", 1: "Attack (1)"})
pd.DataFrame({"Train": tr_bin, "Test": te_bin}).plot(kind="bar", ax=axes[0], color=["#1e40af", "#d97706"], width=0.6)
axes[0].set_title("Binary Label Distribution (Train vs Test)")
axes[0].set_ylabel("Record Count")

cat_df = pd.DataFrame({
    "Train": df_train[TARGET_MULTI].value_counts(),
    "Test": df_test[TARGET_MULTI].value_counts()
}).fillna(0).sort_values(by="Train", ascending=True)
cat_df.plot(kind="barh", ax=axes[1], color=["#2563eb", "#f59e0b"], width=0.7)
axes[1].set_title("Attack Category Distribution (Log Scale)")
axes[1].set_xscale("log")
plt.tight_layout()
plt.show()

## 3. Categorical Feature Distributions & Domain Shift
Inspecting the categorical features: `proto`, `service`, and `state`. Checking for unseen categories in test.

In [ ]:
cat_stats = analyze_categorical_features(df_train, df_test)
for col, info in cat_stats.items():
    print(f"Column: {col}")
    print(f"  Unique in Train: {info['train_unique_count']} | Unique in Test: {info['test_unique_count']}")
    print(f"  Test-only categories: {info['test_only_values']}")
    print(f"  Train-only categories: {info['train_only_values'][:5]}")
    print()

print("Key Finding: Connection states 'ACC' and 'CLO' appear only in the test set.")
print("Recommendation: One-hot encoders must use handle_unknown='ignore'.")

## 4. Numeric Features, Variance, and Multicollinearity
Analyzing 39 numeric features, checking for low-variance features and collinear clusters.

In [ ]:
num_stats = analyze_numeric_features(df_train)
print("Lowest variance features:")
for k, v in num_stats["lowest_variance_features"].items():
    print(f"  {k}: {v}")

corr_stats = analyze_correlations(df_train, threshold=0.90)
print(f"\nDetected {corr_stats['high_correlation_pairs_count']} pairs with |r| >= 0.90:")
for p in corr_stats["high_correlation_pairs"][:8]:
    print(f"  {p['feature_1']} <-> {p['feature_2']}: r = {p['correlation']}")

## 5. Rare Attack Categories Deep Dive
Focusing on attack classes with < 2% prevalence: `Analysis`, `Backdoor`, `Shellcode`, `Worms`.

In [ ]:
rare_stats = analyze_rare_categories(df_train, df_test)
for cat, data in rare_stats.items():
    print(f"{cat:<12}: Train={data['train_count']:>5} ({data['train_pct']:>5.2f}%) | Test={data['test_count']:>4} ({data['test_pct']:>5.2f}%)")

print("\nImplication: Multiclass models must employ class-weighted loss functions or cost-sensitive metrics (macro F1-score) to prevent ignoring these rare threats.")